# 02 — Analysis: figures and hypothesis tests

Everything here is computed by `scripts/make_figures.py` (this notebook calls it, so the two cannot disagree).
Inputs: `results/{main_grid,n_sweep,big_qubits}/runs.jsonl`, `results/noise.csv`, `results/hardware/*.json`,
and the ATLAS CSV (Fig. 5 only). Outputs: `results/figures/*.{pdf,png}`, `results/summary_table.csv`,
`results/h2_analysis.json`.

**Hypotheses** (not stated formally in the repo before the grid was run, so these tests are *exploratory*):

| | statement |
|---|---|
| H1 | a quantum kernel (fidelity or projected) is competitive with, or beats, tuned classical models |
| H2 | the geometric difference g(K_C ‖ K_Q) predicts the AUC gap (quantum − classical) |
| H3 | kernel concentration — off-diagonal variance of K_Q shrinking with qubits — degrades the quantum model |
| H4 | at fixed n, a smaller bandwidth c avoids concentration and restores AUC |

**Significance rule.** A seed-paired difference whose mean is within one seed-to-seed sd is *not significant*
(with 5 seeds this is lenient: |mean| > 1 sd ≈ paired t 2.2, p ≈ 0.09). Correlations: 95% bootstrap CI over
configurations; a CI containing 0 is not significant. Configurations share data seeds, so these CIs are optimistic.

**Fairness.** The quantum bandwidth is chosen per seed by CV AUC. `best_classical_auc` is the max of three
models' *test* AUCs (flatters classical), so H1 is judged against the CV-tuned RBF-SVM; best-of-three is shown too.

In [1]:
import json, sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'scripts'))
sys.path.insert(0, str(ROOT))
import make_figures as mf

mf.main()   # writes figures, summary_table.csv and h2_analysis.json
report = json.loads((ROOT / 'results' / 'h2_analysis.json').read_text())
table = pd.read_csv(ROOT / 'results' / 'summary_table.csv')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
print(len(table), 'configurations in summary_table.csv')

wrote results/summary_table.csv (95 configurations), results/h2_analysis.json and 12 figure files in results\figures
H1: no: no reliable advantage; competitive only at small n -- 2 of 34 (model, map, n) combinations beat the RBF-SVM by more than the seed spread (fidelity QSVM ZZ map, reps 1 n=2: +0.0066 ± 0.0044, fidelity QSVM ZZ map, reps 2 n=2: +0.0077 ± 0.0038); ~1.5 such positive hits are expected by chance under a true gap of 0 with this lenient 1-sd rule; 0 beat the best of the three classical models. 17 of 34 are not significantly worse than the RBF-SVM (fidelity QSVM: 10 of 17), mostly the Z map and n <= 4; ZZ maps fall significantly behind from n = 6 on
H2: no: g does not predict the gap in the hypothesised direction -- (pooled, lambda=0.01) rho(g, gap) = -0.44 [-0.61, -0.22]; (b) partial | log var = -0.66 [-0.75, -0.54]; (a) within map: Z map, reps 2 -0.77 [-0.90, -0.56], ZZ map, reps 1 -0.27 [-0.66, 0.21], ZZ map, reps 2 -0.06 [-0.45, 0.40]; lambda=0.1 pooled 0.34 [0.09, 0.5

## H1 — quantum vs classical AUC

![Fig. 1](../results/figures/fig1_auc_vs_qubits.png)

In [2]:
h1 = pd.DataFrame(report['H1'])
cols = ['model', 'feature_map', 'n_qubits', 'bandwidths_searched', 'auc', 'gap_vs_rbf', 'gap_vs_rbf_sd',
        'gap_vs_rbf_label', 'gap_vs_best', 'gap_vs_best_label']
h1[cols].round(4)

,model,feature_map,n_qubits,bandwidths_searched,auc,gap_vs_rbf,gap_vs_rbf_sd,gap_vs_rbf_label,gap_vs_best,gap_vs_best_label
0,fidelity QSVM,"Z map, reps 2",2,5,0.7337,0.0019,0.0071,not significant,-0.0036,not significant
1,fidelity QSVM,"Z map, reps 2",4,5,0.7895,0.0029,0.0052,not significant,0.0019,not significant
2,fidelity QSVM,"Z map, reps 2",6,5,0.7976,0.0041,0.0104,not significant,-0.0001,not significant
3,fidelity QSVM,"Z map, reps 2",8,5,0.8094,-0.0054,0.0096,not significant,-0.0076,not significant
4,fidelity QSVM,"Z map, reps 2",10,5,0.8170,-0.0049,0.0148,not significant,-0.0051,not significant
5,fidelity QSVM,"ZZ map, reps 1",2,5,0.7384,0.0066,0.0044,quantum better,0.0011,not significant
6,fidelity QSVM,"ZZ map, reps 1",4,5,0.7884,0.0018,0.0075,not significant,0.0008,not significant
7,fidelity QSVM,"ZZ map, reps 1",6,5,0.7851,-0.0084,0.0123,not significant,-0.0126,classical better
8,fidelity QSVM,"ZZ map, reps 1",8,5,0.8020,-0.0128,0.0090,classical better,-0.0150,classical better
9,fidelity QSVM,"ZZ map, reps 1",10,5,0.7995,-0.0224,0.0165,classical better,-0.0226,classical better


## H2 — does g predict the gap?

![Fig. 2](../results/figures/fig2_gap_vs_g.png)

Pooled, (a) within each feature map, (b) partial Spearman controlling for log kernel variance.

In [3]:
h2 = report['H2']
fmt = lambda d: f"{d['rho']:+.2f} [{d['ci_low']:+.2f}, {d['ci_high']:+.2f}] {'' if d['label'] == 'significant' else '(n.s.)'}"
rows = []
for lam in ['lambda=0', 'lambda=0.001', 'lambda=0.01', 'lambda=0.1']:
    r = {'lambda': lam, 'pooled': fmt(h2['pooled'][lam]), 'partial | log var': fmt(h2['partial_controlling_log_var'][lam])}
    for fm, d in h2['within_feature_map'][lam].items():
        r[f'within {fm}'] = fmt(d)
    for fm, d in h2['within_feature_map_partial'][lam].items():
        r[f'within+partial {fm}'] = fmt(d)
    rows.append(r)
pd.DataFrame(rows).set_index('lambda').T

lambda,lambda=0,lambda=0.001,lambda=0.01,lambda=0.1
pooled,"-0.50 [-0.65, -0.32]","-0.63 [-0.74, -0.46]","-0.44 [-0.61, -0.22]","+0.34 [+0.09, +0.54]"
partial | log var,"-0.72 [-0.82, -0.59]","-0.77 [-0.84, -0.67]","-0.66 [-0.75, -0.54]","+0.03 [-0.19, +0.25] (n.s.)"
"within Z map, reps 2","-0.62 [-0.86, -0.29]","-0.79 [-0.92, -0.57]","-0.77 [-0.90, -0.56]","-0.49 [-0.77, -0.10]"
"within ZZ map, reps 1","-0.37 [-0.65, -0.01]","-0.43 [-0.69, -0.01]","-0.27 [-0.66, +0.21] (n.s.)","+0.50 [+0.07, +0.78]"
"within ZZ map, reps 2","-0.25 [-0.55, +0.12] (n.s.)","-0.37 [-0.66, +0.06] (n.s.)","-0.06 [-0.45, +0.40] (n.s.)","+0.59 [+0.23, +0.81]"
"within+partial Z map, reps 2","-0.82 [-0.92, -0.59]","-0.89 [-0.95, -0.70]","-0.84 [-0.93, -0.62]","-0.48 [-0.77, -0.08]"
"within+partial ZZ map, reps 1","-0.62 [-0.81, -0.27]","-0.59 [-0.77, -0.28]","-0.49 [-0.73, -0.14]","-0.31 [-0.56, +0.15] (n.s.)"
"within+partial ZZ map, reps 2","-0.66 [-0.84, -0.28]","-0.66 [-0.82, -0.38]","-0.61 [-0.79, -0.34]","-0.40 [-0.61, -0.05]"


(c) Huang et al.'s label-aware criterion, log10(s_C / s_Q), compared with g. H2 expects ρ > 0 for both.

In [4]:
la = h2['label_aware']
print('pooled                 ', fmt(la['pooled']))
print('partial | log var      ', fmt(la['partial_controlling_log_var']))
for fm, d in la['within_feature_map'].items():
    print(f'within {fm:16s}', fmt(d))
for lam, d in la['ratio_minus_g'].items():
    print(f"rho(ratio) - rho(g, {lam:12s}) = {d['diff']:+.2f} [{d['ci_low']:+.2f}, {d['ci_high']:+.2f}] {d['label']}")
print()
print('classification AUC (competitive vs not):', {k: (round(v, 3) if isinstance(v, float) else v) for k, v in h2['classification'].items()})
print()
br = la['bound_regime']
print({k: v for k, v in br.items() if k not in ('spearman_s_quantum_vs_qsvm_auc', 'interpretation')})
print('rho(s_quantum, QSVM AUC) =', fmt(br['spearman_s_quantum_vs_qsvm_auc']))
print(br['interpretation'])

pooled                  -0.63 [-0.75, -0.48] 
partial | log var       -0.61 [-0.73, -0.44] 
within Z map, reps 2    -0.51 [-0.77, -0.12] 
within ZZ map, reps 1   -0.56 [-0.81, -0.22] 
within ZZ map, reps 2   -0.64 [-0.82, -0.33] 
rho(ratio) - rho(g, lambda=0    ) = -0.13 [-0.29, +0.02] not significant
rho(ratio) - rho(g, lambda=0.001) = -0.00 [-0.15, +0.14] not significant
rho(ratio) - rho(g, lambda=0.01 ) = -0.19 [-0.40, -0.00] significant
rho(ratio) - rho(g, lambda=0.1  ) = -0.97 [-1.22, -0.65] significant

classification AUC (competitive vs not): {'definition': 'AUC for separating configurations whose gap is not significantly negative', 'n_competitive': 21, 'n_total': 79, 'g_lam0.01': 0.209, 'g_lam0': 0.259, 'log_s_ratio': 0.25, 'log_var': 0.632}

{'s_classical_min': 4.391080373036189, 's_classical_median': 36.642649293002215, 'fraction_s_classical_below_1': 0.0, 'fraction_s_quantum_below_1': 0.22784810126582278, 'median_s_quantum_when_var_below_1e-3': 1.0672896432128627}
rho(s_quan

## H3 — concentration vs qubits

![Fig. 3](../results/figures/fig3_concentration_vs_qubits.png)

![Fig. 5](../results/figures/fig5_kernel_heatmaps.png)

In [5]:
h3 = report['H3']
display(pd.DataFrame(h3['variance_slope_vs_qubits']).round(3))
print('rho(log var, QSVM AUC) =', fmt(h3['spearman_log_var_vs_qsvm_auc']))
for k, v in h3['low_variance_regimes'].items():
    print(k, '->', {a: b for a, b in v.items() if a != 'configs'})
print(h3['note'])

,feature_map,bandwidth,n_range,log10_var_per_qubit,slope_ci95,r2,label
0,"Z map, reps 2",0.05,2-10,0.080,0.021,0.946,increasing
1,"Z map, reps 2",0.10,2-10,0.070,0.021,0.934,increasing
2,"Z map, reps 2",0.25,2-10,0.014,0.010,0.720,increasing
3,"Z map, reps 2",0.50,2-10,-0.105,0.010,0.993,decreasing
4,"Z map, reps 2",1.00,2-10,-0.202,0.009,0.999,decreasing
5,"ZZ map, reps 1",0.05,2-10,0.044,0.030,0.728,increasing
6,"ZZ map, reps 1",0.10,2-10,-0.059,0.019,0.927,decreasing
7,"ZZ map, reps 1",0.25,2-10,-0.275,0.030,0.991,decreasing
8,"ZZ map, reps 1",0.50,2-10,-0.424,0.025,0.997,decreasing
9,"ZZ map, reps 1",1.00,2-10,-0.481,0.012,0.999,decreasing


rho(log var, QSVM AUC) = +0.40 [+0.18, +0.58] 
concentrated: var < 1e-3 and off-diagonal mean < 0.1 -> {'n_configs': 17, 'qsvm_auc_median': 0.5499031999999999, 'qsvm_auc_max': 0.7202648}
near-constant: var < 1e-3 and off-diagonal mean > 0.5 -> {'n_configs': 5, 'qsvm_auc_median': 0.781716, 'qsvm_auc_max': 0.8035128}
off-diagonal variance alone does not identify failure: nearly constant kernels (Z map, small c) also have small variance but still classify well in exact simulation; they would be fragile under shot noise, which the noise study did not test


## H4 — bandwidth at fixed n

![Fig. 4](../results/figures/fig4_vs_bandwidth.png)

In [6]:
pd.DataFrame(report['H4']).round(3)

,n_qubits,feature_map,c_best_by_cv,auc_at_c_best,auc_at_c1,auc_gain_vs_c1,auc_gain_sd,auc_gain_label,log10_var_at_c_best,log10_var_at_c1,c_with_max_g_lam0.01,c_with_max_auc_is_c_with_max_g
0,4,"Z map, reps 2",0.25,0.788,0.670,0.118,0.044,smaller c better,-1.610,-1.475,1.00,False
1,4,"ZZ map, reps 1",0.05,0.789,0.493,0.296,0.029,smaller c better,-1.468,-2.203,0.25,False
2,4,"ZZ map, reps 2",0.05,0.785,0.519,0.266,0.020,smaller c better,-1.261,-2.159,0.25,False
3,8,"Z map, reps 2",0.25,0.813,0.707,0.106,0.014,smaller c better,-1.568,-2.332,1.00,False
4,8,"ZZ map, reps 1",0.05,0.804,0.499,0.305,0.017,smaller c better,-1.347,-4.148,0.25,False
5,8,"ZZ map, reps 2",0.05,0.796,0.511,0.285,0.025,smaller c better,-1.276,-4.087,0.10,False


## Noise and hardware

![Fig. 6](../results/figures/fig6_noise_and_hardware.png)

Details in `results/noise_summary.md`. The hardware AUCs use 24 test events and are demonstrations, not
statistically strong numbers.

## Verdicts

In [7]:
for k, v in report['verdicts'].items():
    if isinstance(v, str):
        print(k + ':', v)
    else:
        print(k + ':', v['held'])
        print('    ' + v['detail'])
        print()

H1: no: no reliable advantage; competitive only at small n
    2 of 34 (model, map, n) combinations beat the RBF-SVM by more than the seed spread (fidelity QSVM ZZ map, reps 1 n=2: +0.0066 ± 0.0044, fidelity QSVM ZZ map, reps 2 n=2: +0.0077 ± 0.0038); ~1.5 such positive hits are expected by chance under a true gap of 0 with this lenient 1-sd rule; 0 beat the best of the three classical models. 17 of 34 are not significantly worse than the RBF-SVM (fidelity QSVM: 10 of 17), mostly the Z map and n <= 4; ZZ maps fall significantly behind from n = 6 on

H2: no: g does not predict the gap in the hypothesised direction
    (pooled, lambda=0.01) rho(g, gap) = -0.44 [-0.61, -0.22]; (b) partial | log var = -0.66 [-0.75, -0.54]; (a) within map: Z map, reps 2 -0.77 [-0.90, -0.56], ZZ map, reps 1 -0.27 [-0.66, 0.21], ZZ map, reps 2 -0.06 [-0.45, 0.40]; lambda=0.1 pooled 0.34 [0.09, 0.54] but partial | log var 0.03 [-0.19, 0.25], i.e. explained by kernel variance; (c) label-aware log(s_C/s_Q): pool